## Fake News Detection System

In [36]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import re
import warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

In [37]:
current_folder = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_path = project_root/"dataset"
df = pd.read_csv(data_path/"news_data.csv")

/tmp/ipykernel_38229/480025893.py:9: DtypeWarning: Columns (0: Unnamed: 22, 1: Unnamed: 23, 2: Unnamed: 24, 3: Unnamed: 25, 4: Unnamed: 26, 5: Unnamed: 27, 6: Unnamed: 28, 7: Unnamed: 29, 8: Unnamed: 30, 9: Unnamed: 31, 10: Unnamed: 32, 11: Unnamed: 33, 12: Unnamed: 34, 13: Unnamed: 35, 14: Unnamed: 36, 15: Unnamed: 37, 16: Unnamed: 38, 17: Unnamed: 39, 18: Unnamed: 40, 19: Unnamed: 41, 20: Unnamed: 42, 21: Unnamed: 43, 22: Unnamed: 44, 23: Unnamed: 45, 24: Unnamed: 46, 25: Unnamed: 47, 26: Unnamed: 48, 27: Unnamed: 49, 28: Unnamed: 50, 29: Unnamed: 51, 30: Unnamed: 52, 31: Unnamed: 53, 32: Unnamed: 54, 33: Unnamed: 55, 34: Unnamed: 56, 35: Unnamed: 57, 36: Unnamed: 58, 37: Unnamed: 59, 38: Unnamed: 60, 39: Unnamed: 61, 40: Unnamed: 62, 41: Unnamed: 63, 42: Unnamed: 64, 43: Unnamed: 65, 44: Unnamed: 66, 45: Unnamed: 67, 46: Unnamed: 68, 47: Unnamed: 69, 48: Unnamed: 70, 49: Unnamed: 71, 50: Unnamed: 72, 51: Unnamed: 73, 52: Unnamed: 74, 53: Unnamed: 75, 54: Unnamed: 76, 55: Unnamed: 77

In [38]:
df.head(3)

,text,label,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U.S. Secretary of State John F. Kerry said Mon...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df.shape

(7795, 139)

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Columns: 139 entries, text to Unnamed: 138
dtypes: str(139)
memory usage: 8.3 MB


In [41]:
df['label'].value_counts()

label
REAL                                                                                                                  3161
FAKE                                                                                                                  3154
 or naturalization or by jus sanguinis – inherited through ancestors/parents                                             3
 for example                                                                                                             2
 etc.                                                                                                                    2
                                                                                                                      ... 
 culminating with the Siege of Jerusalem in 70 CE. In viscous hand to hand combat between the various Jewish sects       1
 after all                                                                                                               1
 Fla. Bide

### Too many blank columns. Lets filter this out

### Found some unusual label for fake news detection. It seems there are some titles or news crept in label.

In [42]:
df = df[['text', 'label']]

df = df[df['label'].isin(valid_labels)]

In [43]:
df.describe()

,text,label
count,6315,6315
unique,6040,2
top,"Killing Obama administration rules, dismantlin...",REAL
freq,58,3161


In [44]:
df['label'].value_counts()

label
REAL    3161
FAKE    3154
Name: count, dtype: int64

## Descriptive Statistical Report
- We found unique news count is 6040. Most frequent news comes 58 times. Lets take a look at duplicates.

In [45]:
print(df.duplicated().sum())

275


In [46]:
df[df.duplicated()]

,text,label
61,"On this day in 1973, J. Fred Buzhardt, a lawye...",REAL
90,"Killing Obama administration rules, dismantlin...",REAL
157,"Killing Obama administration rules, dismantlin...",REAL
233,"On this day in 1973, J. Fred Buzhardt, a lawye...",REAL
410,"Killing Obama administration rules, dismantlin...",REAL
...,...,...
7687,Email \nISIS barbarians used an industrial dou...,FAKE
7693,Email \nNorth Korea’s Foreign Ministry slammed...,FAKE
7710,A verdict in 2017 could have sweeping conseque...,REAL
7730,"Killing Obama administration rules, dismantlin...",REAL


In [47]:
df[df.duplicated()].index

Index([  61,   90,  157,  233,  410,  417,  569,  596,  647,  655,
       ...
       7608, 7628, 7635, 7654, 7660, 7687, 7693, 7710, 7730, 7788],
      dtype='int64', length=275)

### Drop Duplicate news

In [48]:
df = df.drop_duplicates(keep='first')

In [49]:
df.describe()

,text,label
count,6040,6040
unique,6040,2
top,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
freq,1,3061
